# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ujjwalupreti/flyrank-internship-capstone/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1 – Freshness Multiplier

**Methodology question**

* The paper reports that refreshed mature pages showed substantially higher health and impressions. How were the refreshed pages selected? Could editorial selection bias contribute to this observed difference? A matched comparison or time-aware validation would strengthen this finding.

## Finding 2 – Click Capture by Position Tier

**Methodology question**

* The paper shows weighted CTR by position tier. Were minimum impression thresholds applied to every bucket? Reporting the sample size (n) and the volume floor for each tier would help readers assess the stability of the estimates.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5, I evaluated my model using a random train/test split. In this notebook, I re-evaluate the same model using a GroupShuffleSplit based on `client_id`. This provides a more realistic estimate of how well the model generalises to previously unseen clients.

In [4]:
import duckdb
import pandas as pd
from google.colab import userdata

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

con = duckdb.connect()

hf_token = userdata.get("HF_TOKEN")

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

REL_FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
REL_DIM = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

df = con.sql(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position,
    SUM(ga4_sessions) AS sessions,
    SUM(ga4_pageviews) AS pageviews,
    SUM(ga4_engaged_sessions) AS engaged_sessions,
    SUM(scroll_events) AS scroll_events,

    MAX(d.search_volume) AS search_volume,
    MAX(d.word_count) AS word_count,
    MAX(d.char_count) AS char_count,
    MAX(d.content_type) AS content_type,

    CASE
        WHEN SUM(gsc_clicks) = 0 THEN 0
        ELSE 1
    END AS target

FROM {REL_FACT} f
JOIN {REL_DIM} d
ON f.content_hash_id = d.content_hash_id

GROUP BY
    f.client_hash_id,
    f.content_hash_id
""").to_df()

groups = df["client_hash_id"]

y = df["target"]

X = df.drop(columns=[
    "target",
    "client_hash_id",
    "content_hash_id"
])

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

model.fit(X_train, y_train)

pred_random = model.predict(X_test)

precision_random = precision_score(y_test, pred_random)
base_rate_random = y_test.mean()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train_g = X.iloc[train_idx]
X_test_g = X.iloc[test_idx]

y_train_g = y.iloc[train_idx]
y_test_g = y.iloc[test_idx]

model.fit(X_train_g, y_train_g)

pred_group = model.predict(X_test_g)

precision_group = precision_score(y_test_g, pred_group)
base_rate_group = y_test_g.mean()

results = pd.DataFrame({
    "Validation Split": ["Random Split", "Grouped Split"],
    "Precision": [
        round(precision_random, 4),
        round(precision_group, 4)
    ],
    "Base Rate": [
        round(base_rate_random, 4),
        round(base_rate_group, 4)
    ]
})

print("=" * 70)
print("Model Validation Comparison")
print("=" * 70)

display(results)

print("\nInterpretation")
print("-" * 70)

if precision_group < precision_random:
    print(
        "The grouped split produced a lower precision than the random split. "
        "This indicates that grouped validation provides a more realistic estimate "
        "of model performance on unseen clients."
    )
else:
    print(
        "The grouped split achieved similar performance to the random split, "
        "indicating that the model generalises well across different clients."
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model Validation Comparison


,Validation Split,Precision,Base Rate
0,Random Split,1.0,0.2077
1,Grouped Split,1.0,0.2106



Interpretation
----------------------------------------------------------------------
The grouped split achieved similar performance to the random split, indicating that the model generalises well across different clients.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

| Potential Leakage | Present? | Action                 |
| ----------------- | -------- | ---------------------- |
| `trend_direction` | No       | Removed                |
| `trend_pct`       | No       | Removed                |
| Product flags     | No       | Not used               |
| Future window     | No       | Checked                |
| IDs as features   | No       | Used only for grouping |

No label-derived features or product-derived scores were included as model inputs. The model uses only observable search-performance signals available before prediction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In this dataset, the Random Forest model ranked content review opportunities more effectively than the rule-based baseline on the selected evaluation metrics. These results are observational and intended to support editorial prioritisation rather than make causal claims.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.